# Dynamic Personality Vector — Experiment Runner
EXPERIMENTS 리스트에 파일 경로와 interval을 넣고 **Run All** 하세요.
모델은 한 번만 로드하고 전체 실험에서 재사용합니다.

In [ ]:
!mkdir -p src
!pip install -e . --no-build-isolation

In [ ]:
# ══════════════════════════════════════════════════════
# ★ 실행할 파일 목록 — 여기만 수정하세요
# ══════════════════════════════════════════════════════

BASE = "/workspace/auto_personality/event_context"

# EXPERIMENTS = [
#     ── Case 1 (단일 사건, 20턴) ─────────────────────
#     (f"{BASE}/context_case1_birth.txt",            5),
#     (f"{BASE}/context_case1_divorce.txt",          5),
#     (f"{BASE}/context_case1_first_job.txt",        5),
#     (f"{BASE}/context_case1_graduation.txt",       5),
#     (f"{BASE}/context_case1_marriage.txt",         5),
#     (f"{BASE}/context_case1_new_relationship.txt", 5),
#     (f"{BASE}/context_case1_separation.txt",       5),
#     (f"{BASE}/context_case1_unemployment.txt",     5),

#     ── Case 2 (복합 사건, 25턴) ─────────────────────
#     (f"{BASE}/context_case2_first_job_birth.txt",            5),
#     (f"{BASE}/context_case2_first_job_divorce.txt",           5),
#     (f"{BASE}/context_case2_graduation_new_relationship.txt", 5),
#     (f"{BASE}/context_case2_marriage_first_job.txt",          5),
#     (f"{BASE}/context_case2_unemployment_divorce.txt",        5),

#     ── Case 3 (3중 복합, 50턴) ──────────────────────
#     (f"{BASE}/context_case3_birth_unemployment_divorce.txt",               5),
#     (f"{BASE}/context_case3_first_job_birth_new_relationship.txt", 5),
#     (f"{BASE}/context_case3_first_job_divorce_graduation.txt",                5),
#     (f"{BASE}/context_case3_unemployment_separation_divorce.txt",          5),
#     (f"{BASE}/context_case3_new_relationship_marriage_graduation.txt",                5),
# ]

EXPERIMENTS = [
    # ── Case 1: 단일 사건의 순수 영향력 측정 (Baseline) ──
    (f"{BASE}/context_case1_birth.txt", 5),     # [긍정/변화] 출산: 삶의 패턴이 크게 바뀌는 단일 사건
    (f"{BASE}/context_case1_divorce.txt", 5),   # [부정/충격] 이혼: BFI(특히 신경증/우울) 변화를 볼 강력한 부정 사건

    # ── Case 2: 이중 사건의 시너지 효과 (Interaction) ──
    (f"{BASE}/context_case2_marriage_first_job.txt", 5), # [복합 긍정] 결혼+취업: 안정감(성실성, 원만성) 상승 관찰용
    (f"{BASE}/context_case2_unemployment_divorce.txt", 5),# [복합 부정] 실직+이혼: 연속된 스트레스 하에서의 페르소나 붕괴/회복력 관찰용

    #── Case 3: 장기 서사 모델링 (Long-term drift) ──
    (f"{BASE}/context_case3_new_relationship_marriage_graduation.txt", 5), # [3중 복합] 연애+결혼+졸업: 긴 맥락(50턴)에서의 성격 진화
]

print(f"총 {len(EXPERIMENTS)}개 실험 등록")
from pathlib import Path
for i, (f, iv) in enumerate(EXPERIMENTS, 1):
    print(f"  {i:2d}. [{iv}턴] {Path(f).name}")

In [ ]:
%load_ext autoreload
%autoreload 2
import importlib
import run_pipeline
import merger
import dynamic_alpha

importlib.reload(merger)

import sys
if "merger" in sys.modules:
    del sys.modules["merger"]

importlib.reload(run_pipeline)

if "run_pipeline" in sys.modules:
    del sys.modules["run_pipeline"]
    
importlib.reload(dynamic_alpha)

In [ ]:
# ══════════════════════════════════════════════════════
# ★ 모델 한 번만 로드
# ══════════════════════════════════════════════════════

import sys

sys.path.insert(0, "")

import run_pipeline
from merger import DynamicMerger

shared_merger = DynamicMerger(
    base_model_path=run_pipeline.BASE_MODEL_PATH,
    phi_dir=run_pipeline.PHI_VECTOR_DIR,
    method=run_pipeline.MERGE_METHOD,
    dare_drop_rate=0.5, dare_rescale=True,
    dare_strategy="random", ties_trim_rate=0.7, scaling_coefficient=1.0,
)

print("✅ 모델 로드 완료 — 전체 실험에서 재사용")

In [ ]:
# ══════════════════════════════════════════════════════
# ★ 전체 실험 실행 (수정본)
# ══════════════════════════════════════════════════════

from pathlib import Path
import run_pipeline
import merger
import dynamic_alpha

success, fail = [], []
NUM_RUNS = 5  # 각 시나리오당 반복 횟수 설정

for idx, (ctx_file, interval) in enumerate(EXPERIMENTS, 1):
    print(f"\n{'='*60}")
    print(f"[{idx}/{len(EXPERIMENTS)}] {Path(ctx_file).name} (interval={interval}턴)")
    print(f"{'='*60}")

    # 1. 시나리오 설정값 적용
    run_pipeline.CONTEXTS_FILE  = ctx_file
    run_pipeline.EVENT_INTERVAL = interval

    # 2. 각 시나리오를 5번 반복 실행
    for run_id in range(1, NUM_RUNS + 1):
        print(f"  ▶ [반복 {run_id}/{NUM_RUNS}] 실행 시작...")
        
        try:
            # 💡 핵심: run_pipeline 내부의 RUN_ID를 현재 루프 번호로 업데이트
            run_pipeline.RUN_ID = run_id 

            # 모델 재사용(Shared Merger)을 통해 불필요한 머징 시간 단축
            run_pipeline.main(merger=shared_merger)
            
            success.append(f"{Path(ctx_file).name}_run{run_id}")

        except Exception as e:
            print(f"  ❌ 반복 {run_id} 실패: {e}")
            import traceback; traceback.print_exc()
            fail.append(f"{Path(ctx_file).name}_run{run_id}")

print(f"\n{'='*60}")
print(f"완료: {len(success)}/{len(EXPERIMENTS) * NUM_RUNS} 성공")
if fail:
    print("실패한 실험 목록:")
    for f in fail:
        print(f"  ❌ {f}")

In [ ]:
from dynamic_alpha import detect_life_event

test = '[separation] Sarah liked a guy named Saul.'
result = detect_life_event(test)
print(result)